[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aranaur/aranaur.rbind.io/blob/main/lectures/kse/MATH840/26autumn/labs/_demo10.ipynb)

# MATH840 — Week 10 practice: foundation models against everything you know

**Not marked.** Ten weeks ago you started with the seasonal naive forecast. Since then: ETS, ARIMA,
Prophet, combinations, neural networks. Today the newest arrivals — **models that have never seen your
series and are not trained on it at all** — meet all of them, on the same windows and by the same rules.

| | Story | The question | Min |
|---|---|---|---:|
| 1 | ⚡ Melbourne's electricity: a rematch | can a model with no training beat last week's tuned Prophet? | 20 |
| 2 | 🏆 300 series from the M3 competition | who wins across many series — and at what cost? | 20 |
| 3 | 🎯 the same 300 series | whose 80% interval is honest? | 10 |
| 4 | 📏 short histories | how much past does each model need? | 10 |
| 5 | 🍽️ one series, everything | tonight's lab, done on a public series | 20 |

🔮 marks a guess: write it down before running the cell. ❓ marks a question for the room.

**Run the notebook before the session** (`Runtime → Run all`, about half an hour on Colab's free CPU):
the models download about 1.5 GB on first use.

## 0. Setup

In [ ]:
!pip install -q statsforecast==2.1.1 neuralforecast==3.3.0 chronos-forecasting==2.3.2 "timesfm[torch]==3.0.2" tirex-ts==1.4.2 great_tables

import os
os.environ["NIXTLA_ID_AS_COL"] = "true"
import logging
import time
import warnings
warnings.filterwarnings("ignore")
for name in ("cmdstanpy", "prophet", "lightning", "lightning.pytorch", "pytorch_lightning", "transformers"):
    logging.getLogger(name).setLevel(logging.CRITICAL)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from great_tables import GT, loc, style
from matplotlib.colors import LinearSegmentedColormap, to_hex
from prophet import Prophet
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, MSTL, SeasonalNaive

DEVICE = "cpu"              # "cuda" on a GPU runtime; every timing below is for this device
plt.rcParams.update({"figure.figsize": (11, 3.6), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 120, "display.precision", 3)
BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data")
COLOURS = {"actual": "#314f4f", "fm": "#e64173", "old": "#9aa5a5", "nn": "#6A5ACD", "b": "#20B2AA", "c": "#FFA500"}

### Four foundation models, one interface

Each model gets a list of histories and returns, for every series, the median and the 10% and 90%
quantiles — an 80% interval for free. **No fitting**: the models arrive trained.

| Model | From | Size | Inside |
|---|---|---|---|
| Chronos-Bolt (small) | Amazon, 2024 | 48M | T5 encoder–decoder, patches of the series |
| Chronos-2 | Amazon, 2025 | 120M | encoder with attention across series and covariates |
| TimesFM 2.5 | Google, 2025 | 200M | decoder-only transformer, long output patches |
| TiRex | NX-AI, 2025 | 35M | xLSTM — a recurrent network, not a transformer |

In [ ]:
from chronos import BaseChronosPipeline
import timesfm
import tirex

bolt = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-small", device_map=DEVICE)
chronos2 = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
tfm.model.device, tfm.model.device_count = torch.device(DEVICE), 1      # keep TimesFM on DEVICE too
tfm.model.to(DEVICE)
tfm.compile(timesfm.ForecastConfig(max_context=512, max_horizon=128, normalize_inputs=True,
                                   use_continuous_quantile_head=True, fix_quantile_crossing=True))
tirex_model = tirex.load_model("NX-AI/TiRex", device=DEVICE)


def foundation(name, histories, h):
    # -> (median, 10% quantile, 90% quantile), each an array of shape (number of series, h)
    contexts = [torch.tensor(np.asarray(x, float), dtype=torch.float32) for x in histories]
    if name in ("Chronos-Bolt", "Chronos-2"):
        pipe = bolt if name == "Chronos-Bolt" else chronos2
        q, _ = pipe.predict_quantiles(contexts, prediction_length=h, quantile_levels=[0.1, 0.5, 0.9])
        q = np.stack([np.asarray(a).reshape(-1, h, 3)[0] for a in q])     # Chronos-2 adds a variate axis
        return q[:, :, 1], q[:, :, 0], q[:, :, 2]
    if name == "TimesFM 2.5":
        _, q = tfm.forecast(horizon=h, inputs=[c.numpy() for c in contexts])
        q = np.asarray(q)[:, :h]                                           # mean, then 10%, 20%, ..., 90%
        return q[:, :, 5], q[:, :, 1], q[:, :, 9]
    q, _ = tirex_model.forecast(context=contexts, prediction_length=h)
    q = np.asarray(q)                                                      # 10%, 20%, ..., 90%
    return q[:, :, 4], q[:, :, 0], q[:, :, 8]


FOUNDATION = ["Chronos-Bolt", "Chronos-2", "TimesFM 2.5", "TiRex"]

In [ ]:
_shade = LinearSegmentedColormap.from_list("shade", ["#bfe0b0", "#ffffff", "#f5b8cb"])


def scoreboard(table, rules=None, decimals=2):
    # Green is better, pink is worse, the best in bold. A rule is "lower", "higher" or a target value.
    df = table.to_frame() if isinstance(table, pd.Series) else table.copy()
    df.index, df.columns = df.index.map(str), df.columns.map(str)
    rules = rules or {c: "lower" for c in df.columns}
    gt = (GT(df.reset_index(names=" "), rowname_col=" ")
          .fmt_number(columns=list(df.select_dtypes("number").columns), decimals=decimals)
          .sub_missing(missing_text=""))
    for c, rule in rules.items():
        v = df[c].astype(float)
        bad = v if rule == "lower" else -v if rule == "higher" else (v - rule).abs()
        lo, hi = bad.min(), bad.max()
        for i, b in enumerate(bad):
            if np.isnan(b):
                continue
            look = [style.fill(color=to_hex(_shade((b - lo) / (hi - lo) if hi > lo else 0.0)))]
            gt = gt.tab_style(style=look + ([style.text(weight="bold")] if b == lo else []),
                              locations=loc.body(columns=c, rows=[i]))
    return gt


def barh(means, title, xlabel, colour_of, hatch=lambda name: False):
    # One horizontal bar per model, best at the top.
    means = means.sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 0.36 * len(means) + 1.2))
    bars = ax.barh(means.index, means, color=[colour_of(n) for n in means.index])
    for bar, name in zip(bars, means.index):
        if hatch(name):
            bar.set_hatch("//")
            bar.set_edgecolor("white")
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=9)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    plt.show()

## 1. ⚡ Melbourne's electricity: a rematch

Last week you were the grid operator of Victoria. After an hour of work — two seasons, named holidays,
a window around Christmas, temperature and its square — Prophet reached a mean MASE of **0.54** without
the weather and **0.40** with the temperature that actually happened.

Today a model that has never seen this series gets three lines of code.

In [ ]:
vic = pd.read_csv(f"{BASE}/vic_elec.csv", parse_dates=["ds"])
wide = vic.pivot_table(index="ds", columns="unique_id", values="y")
holiday = vic.query("unique_id == 'Demand'").set_index("ds")["Holiday"].astype(bool)
elec = pd.DataFrame({"y": wide["Demand"].resample("D").sum() / 2e3,      # half-hourly MW -> GWh per day
                     "temp": wide["Temperature"].resample("D").max(),
                     "holiday": holiday.resample("D").max().astype(float)}).reset_index()
elec["unique_id"] = "Victoria"
y1 = elec["y"].to_numpy()
H1 = 28

### 1.1 Three lines, no training

The last four weeks of 2014, Christmas included, from everything before them.

In [ ]:
e = len(y1) - H1
median, low, high = foundation("Chronos-2", [y1[:e]], H1)

days = elec["ds"].iloc[e:]
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(elec["ds"].iloc[e - 56:], y1[e - 56:], color=COLOURS["actual"], linewidth=1.2, label="actual")
ax.plot(days, median[0], color=COLOURS["fm"], linewidth=1.8, label="Chronos-2, median")
ax.fill_between(days, low[0], high[0], color=COLOURS["fm"], alpha=0.15, label="10%–90%")
ax.axvline(days.iloc[0], color="grey", linewidth=0.8)
ax.set_title("Chronos-2, zero-shot: December 2014 from the history alone")
ax.legend(loc="lower left", fontsize=8)
plt.show()

### 1.2 The rematch, on last week's nine windows

The same nine four-week windows as last week, April to December 2014. On one side, last week's best
— rebuilt here — on the other, the four foundation models, plus Chronos-2 given the same covariates
Prophet had: the holiday flag and the temperature (the real one, so ex post, like Prophet's).

🔮 **Guess first.** Will any model with no training reach Prophet's 0.54? Its 0.40 with the weather?

In [ ]:
origins1 = [len(y1) - H1 * k for k in range(9, 0, -1)]
scale1 = np.mean(np.abs(y1[7:origins1[0]] - y1[:origins1[0] - 7]))
holidays = pd.DataFrame({"holiday": "public holiday", "ds": elec.loc[elec["holiday"] == 1, "ds"]})


def prophet_elec(weather):
    def forecast(e):
        model = Prophet(daily_seasonality=False, holidays=holidays)
        if weather:
            for column in ("temp", "temp2"):
                model.add_regressor(column)
        data = elec.assign(temp2=elec["temp"] ** 2)
        model.fit(data.iloc[:e])
        return model.predict(data.iloc[e:e + H1])["yhat"].to_numpy()
    return forecast


def chronos2_with(columns):
    def forecast(e):
        out = chronos2.predict_df(elec.iloc[:e][["unique_id", "ds", "y"] + columns],
                                  future_df=elec.iloc[e:e + H1][["unique_id", "ds"] + columns],
                                  id_column="unique_id", timestamp_column="ds", target="y",
                                  prediction_length=H1, quantile_levels=[0.5])
        return out["predictions"].to_numpy()
    return forecast


forecasters1 = {
    "seasonal naive": lambda e: np.tile(y1[e - 7:e], 4),
    "MSTL (7, 365)": lambda e: MSTL(season_length=[7, 365]).fit(y=y1[:e]).predict(h=H1)["mean"],
    "Prophet + holidays": prophet_elec(False),
    "Prophet + holidays + temperature": prophet_elec(True),
    **{name: (lambda name: lambda e: foundation(name, [y1[:e]], H1)[0][0])(name) for name in FOUNDATION},
    "Chronos-2 + holidays + temperature": chronos2_with(["holiday", "temp"]),
}

scores1, seconds1 = {}, {}
for name, f in forecasters1.items():
    start = time.time()
    scores1[name] = [np.mean(np.abs(y1[e:e + H1] - np.asarray(f(e)))) / scale1 for e in origins1]
    seconds1[name] = time.time() - start
scores1 = pd.DataFrame(scores1, index=[f"{elec['ds'][e]:%d %b}" for e in origins1]).T
scores1["mean"] = scores1.mean(axis=1)
scores1["seconds"] = pd.Series(seconds1)
scoreboard(scores1)

In [ ]:
barh(scores1["mean"], "Pink: foundation models, no training; grey: last week's models; hatched: real future temperature",
     "mean MASE over nine windows (shorter is better)",
     lambda n: COLOURS["fm"] if n.split(" +")[0] in FOUNDATION else COLOURS["old"],
     hatch=lambda n: "temperature" in n)

### 1.3 Too good to be true?

One model beat Prophet **with the real weather** — while seeing nothing but the demand itself. Before
celebrating, do what the lecture's checklist says: **check the corpus.** What each model's own
documentation says it was trained on:

| Model | Training data, as published | Australian electricity demand in it? |
|---|---|---|
| Chronos-Bolt | not disclosed; the model card says the Chronos benchmark II, which includes *Australian Electricity*, was **not** seen | probably not |
| Chronos-2 | technical report, Table 6: 22 real datasets, none Australian | **no** |
| TiRex | paper, Tables 5–6: Chronos and GIFT-Eval subsets, none Australian | no (version 1.0) |
| TimesFM 2.5 | model card: *GiftEvalPretrain*, whose list (GIFT-Eval paper, Table 14) includes Monash's **Australian Electricity Demand** — half-hourly demand of five states, Victoria among them, 2002–2015 — and a second Monash series, *Elecdemand* | **yes** |

Our 2012–2014 daily demand is the same electricity, added up by day.

**❓ Question.** Does this prove that TimesFM "remembered" December 2014? What would you need to see to
tell remembering from being good? And what does it mean for the 0.31?

## 2. 🏆 300 series from the M3 competition

One series is one story — and, as we just saw, maybe a remembered one. Now breadth: 300 monthly series
from the M3 competition (1999), forecast 18 months ahead, the competition's own horizon.

On the M3 corpus check: **M3 is in TimesFM 2.5's GIFT-Eval pretraining list**, and in none of the other
three (Chronos-2 Table 6, TiRex Tables 5–6; Chronos-Bolt's card says the benchmark containing M3 was
unseen). TimesFM's row carries a † below.

The contenders, from Week 1 to today:

- **seasonal naive**, **ETS**, **ARIMA**, **Theta**, and the **average of ETS, ARIMA and Theta** (Week 9's
  lesson);
- **N-BEATS** and **MLP** from Week 9, trained here from scratch on the 300 histories (a global model);
- the four **foundation models**, zero-shot, and the **average of the three newest**.

In [ ]:
m3 = pd.read_csv(f"{BASE}/m3_monthly.csv", parse_dates=["ds"])
ids = np.random.default_rng(1).choice(m3["unique_id"].unique(), 300, replace=False)
panel = m3[m3["unique_id"].isin(ids)].sort_values(["unique_id", "ds"])
H2 = 18
test2 = panel.groupby("unique_id").tail(H2)
train2 = panel.drop(test2.index)
histories2 = [g["y"].to_numpy() for _, g in train2.groupby("unique_id")]
actual2 = np.stack([g["y"].to_numpy() for _, g in test2.groupby("unique_id")])
scale2 = np.array([np.mean(np.abs(x[12:] - x[:-12])) for x in histories2])
print(f"{len(histories2)} series, {min(map(len, histories2))} to {max(map(len, histories2))} months of history")

In [ ]:
point2, band2, seconds2 = {}, {}, {}

start = time.time()
sf = StatsForecast(models=[SeasonalNaive(12, alias="seasonal naive"), AutoETS(season_length=12, alias="ETS"),
                           AutoARIMA(season_length=12, alias="ARIMA"), AutoTheta(season_length=12, alias="Theta")],
                   freq="MS", n_jobs=-1)
classic = sf.forecast(df=train2, h=H2, level=[80]).sort_values(["unique_id", "ds"])
seconds2["ETS, ARIMA, Theta, naive (together)"] = time.time() - start
for name in ["seasonal naive", "ETS", "ARIMA", "Theta"]:
    point2[name] = classic[name].to_numpy().reshape(-1, H2)
    band2[name] = (classic[f"{name}-lo-80"].to_numpy().reshape(-1, H2), classic[f"{name}-hi-80"].to_numpy().reshape(-1, H2))
point2["average: ETS, ARIMA, Theta"] = (point2["ETS"] + point2["ARIMA"] + point2["Theta"]) / 3

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import MLP, NBEATS, LSTM

NEURAL = dict(max_steps=500, scaler_type="robust", random_seed=1, accelerator="gpu" if DEVICE == "cuda" else "cpu",
              devices=1, enable_progress_bar=False, enable_model_summary=False, logger=False)
for name, model in [("N-BEATS (trained here)", NBEATS(h=H2, input_size=36, **NEURAL)),
                    ("MLP (trained here)", MLP(h=H2, input_size=36, **NEURAL))]:
    start = time.time()
    nf = NeuralForecast(models=[model], freq="MS")
    nf.fit(df=train2)
    point2[name] = nf.predict().sort_values(["unique_id", "ds"])[type(model).__name__].to_numpy().reshape(-1, H2)
    seconds2[name] = time.time() - start

for name in FOUNDATION:
    start = time.time()
    median, low, high = foundation(name, histories2, H2)
    seconds2[name] = time.time() - start
    point2[name], band2[name] = median, (low, high)
point2["average: Chronos-2, TimesFM, TiRex"] = (point2["Chronos-2"] + point2["TimesFM 2.5"] + point2["TiRex"]) / 3

🔮 **Guess first.** Rank them. Where do the neural networks trained on these 300 series land, against
models that never saw them? Does the old average of ETS, ARIMA and Theta survive?

In [ ]:
mase2 = pd.DataFrame({name: np.mean(np.abs(actual2 - f), axis=1) / scale2 for name, f in point2.items()})
mase2 = mase2.rename(columns={"TimesFM 2.5": "TimesFM 2.5 †"})
board2 = pd.DataFrame({"mean MASE": mase2.mean(), "median MASE": mase2.median(),
                       "beats ETS": mase2.lt(mase2["ETS"], axis=0).mean(),
                       "worst series": mase2.max()}).sort_values("mean MASE")
scoreboard(board2, rules={"mean MASE": "lower", "median MASE": "lower", "beats ETS": "higher", "worst series": "lower"})

In [ ]:
def family(name):
    if name.startswith(("N-BEATS", "MLP")):
        return COLOURS["nn"]
    return COLOURS["fm"] if any(name.startswith(f) for f in FOUNDATION) or "Chronos-2, TimesFM" in name else COLOURS["old"]


timing = {"seasonal naive": "ETS, ARIMA, Theta, naive (together)", "ETS": "ETS, ARIMA, Theta, naive (together)",
          "ARIMA": "ETS, ARIMA, Theta, naive (together)", "Theta": "ETS, ARIMA, Theta, naive (together)"}
fig, ax = plt.subplots(figsize=(11, 4.4))
for i, name in enumerate(board2.index):
    key = timing.get(name, name.replace(" †", ""))
    if key not in seconds2:
        continue
    ax.scatter(seconds2[key], board2.loc[name, "mean MASE"], s=60, color=family(name), zorder=3)
    ax.annotate(name, (seconds2[key], board2.loc[name, "mean MASE"]), xytext=(6, 4 if i % 2 else -11),
                textcoords="offset points", fontsize=8)
ax.set_xscale("log")
ax.set_xlabel(f"seconds for all 300 series on {DEVICE} (log scale; the four classic models share one run)")
ax.set_ylabel("mean MASE")
ax.set_title("Grey: Weeks 1–9 · purple: neural networks trained here · pink: foundation models, zero-shot")
plt.show()

**❓ Question.** The networks you trained on 300 series lost to models that were never shown them. Is
that a verdict on neural networks?

## 3. 🎯 Whose 80% interval is honest?

A forecast you act on needs a range. ETS, ARIMA and Theta give intervals from their equations (Weeks 6–8);
the foundation models give quantiles they learned. On the same 300 × 18 months: how often does the truth
land inside, and what is the Winkler score (Week 8: width plus a penalty for every miss)?

🔮 **Guess first.** Which ones are too narrow?

In [ ]:
rows = {}
for name, (lo, hi) in band2.items():
    inside = (actual2 >= lo) & (actual2 <= hi)
    winkler = ((hi - lo) + 10 * (np.maximum(lo - actual2, 0) + np.maximum(actual2 - hi, 0))) / scale2[:, None]
    rows[name + (" †" if name == "TimesFM 2.5" else "")] = {
        "coverage": inside.mean(), "coverage, months 1–6": inside[:, :6].mean(),
        "coverage, months 13–18": inside[:, 12:].mean(), "Winkler": winkler.mean()}
intervals2 = pd.DataFrame(rows).T.sort_values("Winkler")
scoreboard(intervals2, rules={"coverage": 0.8, "coverage, months 1–6": 0.8, "coverage, months 13–18": 0.8, "Winkler": "lower"})

## 4. 📏 How much past does each model need?

Real series are often short: a new product, a new store, a new country. ETS has to estimate everything
from what it is given. A foundation model brings its experience with it.

We cut the same 150 series down to their last 24, 36 or 60 months of history and forecast the same 18
months.

🔮 **Guess first.** With only two years of history, who suffers more — ETS or the foundation models?

In [ ]:
subset = slice(0, 150)
rows = []
label_of = lambda keep: f"last {keep} months" if keep else "all history"
for keep in [24, 36, 60, None]:
    cut = [x[-keep:] if keep else x for x in histories2[subset]]
    label = label_of(keep)
    sf_cut = pd.concat([pd.DataFrame({"unique_id": i, "ds": pd.date_range("2000-01-01", periods=len(x), freq="MS"), "y": x})
                        for i, x in enumerate(cut)])
    ets = AutoETS(season_length=12)
    fc = StatsForecast(models=[ets], freq="MS", n_jobs=-1).forecast(df=sf_cut, h=H2).sort_values(["unique_id", "ds"])
    found = {"ETS": fc["AutoETS"].to_numpy().reshape(-1, H2), "seasonal naive": np.stack([np.tile(x[-12:], 2)[:H2] for x in cut])}
    for name in ["Chronos-2", "TimesFM 2.5", "TiRex"]:
        found[name] = foundation(name, cut, H2)[0]
    for name, f in found.items():
        rows.append({"history": label, "model": name,
                     "mean MASE": np.mean(np.mean(np.abs(actual2[subset] - f), axis=1) / scale2[subset])})
context = pd.DataFrame(rows).pivot(index="model", columns="history", values="mean MASE")
context = context[[label_of(keep) for keep in [24, 36, 60, None]]]

fig, ax = plt.subplots(figsize=(10, 3.8))
label_y = context.iloc[:, -1].sort_values()
for i in range(1, len(label_y)):                       # keep the end labels apart
    label_y.iloc[i] = max(label_y.iloc[i], label_y.iloc[i - 1] + 0.018)
for name, row in context.iterrows():
    colour = COLOURS["fm"] if name in FOUNDATION else COLOURS["old"]
    ax.plot(row.index, row.values, marker="o", color=colour, linestyle="--" if name == "seasonal naive" else "-",
            linewidth=2)
    ax.annotate(name, (3, label_y[name]), xytext=(8, 0), textcoords="offset points", fontsize=8, va="center")
ax.set_ylabel("mean MASE, the same 18 months")
ax.set_title("Less history: everyone gets worse, but not equally")
plt.show()
scoreboard(context)

## 5. 🍽️ One series, everything: tonight's lab in miniature

Tonight's lab, done on a public series: Australia's cafés, restaurants and takeaways (Week 9), 2018 as
the test year, four 12-month windows before it.

- **The champion** — the Week 9 combination of ETS, ARIMA and MSTL on the log, which won 2018.
- **Neural networks trained on this one series** — MLP, N-BEATS, LSTM: 430 months of data, each window
  its own training run.
- **Foundation models**, zero-shot. Australian retail turnover is not in any of their published corpora.
- **The champion plus Chronos-2** — an old model and a new one in one average.

The question for the lab: **does anything new beat what you already had?**

In [ ]:
retail = pd.read_csv(f"{BASE}/aus_retail.csv", parse_dates=["Month"])
cafes = (retail[retail["Industry"] == "Cafes, restaurants and takeaway food services"]
         .groupby("Month", as_index=False)["Turnover"].sum().rename(columns={"Month": "ds", "Turnover": "y"}))
cafes["unique_id"] = "cafes"
y5 = cafes["y"].to_numpy()
H5, WINDOWS = 12, 4


def champion(history, h):
    members = [AutoETS(season_length=12), AutoARIMA(season_length=12), MSTL(season_length=12)]
    return np.mean([np.exp(m.fit(y=np.log(history)).predict(h=h)["mean"]) for m in members], axis=0)


def trained_here(model_class):
    # A neural network fitted on this series only, from scratch, every time it is called.
    def forecast(history, h):
        df = cafes.iloc[:len(history)]
        model = model_class(h=h, input_size=36, **(NEURAL | {"max_steps": 300}))
        nf = NeuralForecast(models=[model], freq="MS")
        nf.fit(df=df[["unique_id", "ds", "y"]])
        return nf.predict()[model_class.__name__].to_numpy()
    return forecast


forecasters5 = {
    "champion: ETS + ARIMA + MSTL": champion,
    "MLP (this series)": trained_here(MLP),
    "N-BEATS (this series)": trained_here(NBEATS),
    "LSTM (this series)": trained_here(LSTM),
    **{name: (lambda name: lambda history, h: foundation(name, [history], h)[0][0])(name)
       for name in ["Chronos-2", "TimesFM 2.5", "TiRex"]},
}

🔮 **Guess first.** Four windows, 2014–2017. Will a network trained on 430 months of this very series beat
the champion? Will a foundation model?

In [ ]:
past5 = y5[:-H5]                                      # everything before 2018
scale5 = np.mean(np.abs(past5[12:] - past5[:-12]))
table5, seconds5, window_fc = {}, {}, {}
for name, f in forecasters5.items():
    start, scores = time.time(), []
    for k in range(WINDOWS, 0, -1):
        end = len(past5) - (k - 1) * H5
        fc = np.asarray(f(past5[:end - H5], H5))
        window_fc[(name, k)] = fc
        scores.append(np.mean(np.abs(past5[end - H5:end] - fc)) / scale5)
    table5[name] = scores
    seconds5[name] = time.time() - start
for k in range(WINDOWS, 0, -1):                       # the old and the new in one average
    window_fc[("champion + Chronos-2", k)] = (window_fc[("champion: ETS + ARIMA + MSTL", k)] + window_fc[("Chronos-2", k)]) / 2
table5["champion + Chronos-2"] = [np.mean(np.abs(past5[len(past5) - k * H5:len(past5) - (k - 1) * H5]
                                                 - window_fc[("champion + Chronos-2", k)])) / scale5
                                  for k in range(WINDOWS, 0, -1)]
table5 = pd.DataFrame(table5, index=[str(2018 - k) for k in range(WINDOWS, 0, -1)]).T
table5["mean"] = table5.mean(axis=1)
table5["seconds"] = pd.Series(seconds5)
scoreboard(table5)

🔮 **Guess first.** Now 2018, which nobody has seen. Will the table's winner win again?

In [ ]:
test5 = {name: np.asarray(f(past5, H5)) for name, f in forecasters5.items()}
test5["champion + Chronos-2"] = (test5["champion: ETS + ARIMA + MSTL"] + test5["Chronos-2"]) / 2
mase5 = pd.Series({name: np.mean(np.abs(y5[-H5:] - f)) / scale5 for name, f in test5.items()}, name="MASE in 2018")
result5 = pd.DataFrame({"windows 2014–2017": table5["mean"], "2018": mase5})
display(scoreboard(result5))

fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(cafes["ds"].iloc[-36:], y5[-36:], color=COLOURS["actual"], linewidth=2, label="actual")
for name, colour, style_ in [("champion: ETS + ARIMA + MSTL", COLOURS["old"], "-"), ("N-BEATS (this series)", COLOURS["nn"], "--"),
                             ("Chronos-2", COLOURS["fm"], "-"), ("TimesFM 2.5", COLOURS["c"], "--")]:
    ax.plot(cafes["ds"].iloc[-H5:], test5[name], style_, color=colour, linewidth=1.6, label=name)
ax.axvline(cafes["ds"].iloc[-H5], color="grey", linewidth=0.8)
ax.set_title("2018: the champion, a network trained on the series, and two foundation models")
ax.legend(loc="upper left", fontsize=8)
plt.show()

**❓ Question.** If you were the analyst for this industry and had to pick one forecaster for the next
five years, which would it be — and what would you check every year?

## What to take into tonight's lab

- **A foundation model is one more candidate** — usually a strong one, and cheap to try: a few lines, no
  fitting. Score it on the same rolling windows as everything else.
- **Check the corpus before you believe a score.** A model that has seen your data, or its source, is not
  forecasting zero-shot. Look at the model card and the paper; when they are silent, say so.
- **A neural network trained on one short series** rarely beats ETS or ARIMA: it has a few hundred numbers
  to learn from. What made N-BEATS famous was training across many series.
- **Intervals:** coverage and Winkler, never coverage alone.
- **The old lessons still hold:** choose on validation windows, not on the test; an average of different
  models is a cheap insurance.